In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from BERTopic_model import run_BERTopic_model

c:\Users\alexb\miniconda3\envs\gnome_BERTopic\lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:13: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


Data loading

In [ ]:
# Dataloading
df = pd.read_csv("../../data/NLP_data_advice_fulltext.csv")
docs = list(df["text"])
docs = [doc.replace('\xa0', '') for doc in docs]
classes = list(df["gen"])

Running the model

In [3]:

base_param_dic = {
    "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
    "n_neighbors": 15,
    "n_components": 10,
    "min_dist": 0.0,
    "min_cluster_size": 20,
    "min_df": 3,
    "max_df": 1.0,
    "ngram_range": (1, 3),
    "top_n_words": 5,
    "seed": 74
}

topic_model = run_BERTopic_model(base_param_dic, docs)
topic_model.get_topic_info()

Batches: 100%|██████████| 32/32 [00:03<00:00,  8.10it/s]
2026-02-24 14:34:47,809 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-02-24 14:34:54,433 - BERTopic - Dimensionality - Completed ✓
2026-02-24 14:34:54,434 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-02-24 14:34:54,468 - BERTopic - Cluster - Completed ✓
2026-02-24 14:34:54,471 - BERTopic - Representation - Extracting topics from clusters using representation models.
2026-02-24 14:34:56,318 - BERTopic - Representation - Completed ✓


,Topic,Count,Name,Representation,KeyBERT,MMR,Representative_Docs
0,-1,134,-1_gnome_gnomes_blue_value,"[gnome, gnomes, blue, value, yellow]","[gnomes, gnome, groups gnomes, colors, colours...","[gnome, groups gnomes, colours, red basket, ye...","[There are 8 gnome colours, and 2 basket colou..."
1,0,272,0_mushrooms_gnome_gnomes_try,"[mushrooms, gnome, gnomes, try, colour]","[gnomes mushrooms, coloured gnomes, number mus...","[gnomes mushrooms, coloured gnomes, number mus...",[Pay close attention when a gnome gives you 0 ...
2,1,184,1_gnome_points_gnomes_hat,"[gnome, points, gnomes, hat, colour]","[points gnomes, colour gnomes, colour gnome, c...","[points gnomes, colour gnomes, gnomes, gnome g...",[In most games the gnome with the smaller hat ...
3,2,146,2_basket_red_yellow_mushrooms,"[basket, red, yellow, mushrooms, gnomes]","[gnomes yellow basket, gnomes red basket, bask...","[gnomes yellow basket, gnomes red basket, bask...",[There are two different colours of baskets in...
4,3,99,3_points_blue_colours_pink,"[points, blue, colours, pink, purple]","[certain colours, pick colour, colours, colors...","[certain colours, pick colour, colors, blue gr...","[There are two themes of colours, purple, pink..."
5,4,81,4_basket_red_baskets_yellow,"[basket, red, baskets, yellow, points]","[basket colours, colour basket, basket colour,...","[basket colours, colour basket, yellow baskets...",[On the screen you will be presented with two ...
6,5,34,5_hats_tall_hat_tall hats,"[hats, tall, hat, tall hats, short]","[tall hats, short hat, taller hat, smaller hat...","[tall hats, smaller hat, hats pay, hat colour,...",[The taller hats seem to pay out best but it i...
7,6,28,6_keys_game_breaks_just,"[keys, game, breaks, just, fingers]","[press keys, fingers keys, keyboard, stay focu...","[press keys, stay focused, gnomes, play, scree...",[Keep your hands on the S and K keys and the s...
8,7,22,7_forest_mushrooms_gnomes_green,"[forest, mushrooms, gnomes, green, green yellow]","[gnomes mushrooms, mushrooms forest, mushrooms...","[gnomes mushrooms, mushrooms forest, mushrooms...",[There are two distinct groups of gnomes(Group...


Saving the model

In [ ]:
#topic_model.save(os.path.join(os.path.dirname(os.getcwd()), "results", "BERTopic_model"), serialization="safetensors", save_ctfidf=True, save_embedding_model="all-MiniLM-L6-v2")

Add topic info to the data

In [ ]:
topic_distr, _ = topic_model.approximate_distribution(docs)
df_stat = pd.read_csv("../../data/NLP_data_stake.csv")

for topic_n in range(len(topic_distr[0,:])):
    topic_name = "topic_" + str(topic_n)
    df_stat[topic_name] = topic_distr[:, topic_n]

df_stat["assigned_topic"] = topic_model.topics_

df_stat.head()
parent_topic_weight = []
var_names = [f'topic_{n}' for n in range(len(topic_distr[0, :]))]

for i in range(len(df_stat["ID"])):
    parent_ID = df_stat["parent_ID"][i]
    parent_row = df_stat[df_stat["ID"] == parent_ID]
    if len(parent_row) < 1:
        parent_topic_weight.append([None for n in range(len(topic_distr[0, :]))])
    else:
        res = parent_row[var_names].values.tolist()
        parent_topic_weight.append(res[0])

parent_topic_df = pd.DataFrame(parent_topic_weight)
parent_topic_df.columns = [f'parent_topic_{n}' for n in range(len(topic_distr[0, :]))]

df_stat = pd.concat([df_stat, parent_topic_df], axis=1)
df_stat = df_stat.loc[:, ~df_stat.columns.str.contains('^Unnamed')]

df_stat.tail()

100%|██████████| 1/1 [00:00<00:00,  2.47it/s]


,ID,understanding_score,pl1_understanding,pl2_understanding,score_corrected,rank,filename,binary_score,ADVICE,gen,...,topic_7,assigned_topic,parent_topic_0,parent_topic_1,parent_topic_2,parent_topic_3,parent_topic_4,parent_topic_5,parent_topic_6,parent_topic_7
995,668a935e677dc562a445cd03,0.379820,0.281283,0.478356,0.247881,80,053402_experiment_2024-07-10_19h18.39.977.csv,0.558594,Try to match the gnome colours to the yellow ...,10,...,0.065823,2,0.170212,0.058973,0.338370,0.064418,0.267185,0.007766,0.003326,0.089749
996,668bb99e5fa8b9ebb3c428ac,0.237887,0.201746,0.274028,-0.082329,91,909045_experiment_2024-07-10_18h48.16.190.csv,0.519531,based on the colours of the gnomes - keep chan...,10,...,0.472417,7,0.487939,0.000000,0.270484,0.000000,0.000000,0.000000,0.000000,0.241578
997,668bfbcbfd735b6652c60f20,-0.116250,-0.155149,-0.077350,0.732068,61,489637_experiment_2024-07-10_14h22.18.159.csv,0.542969,"hello, there are two different colors of baske...",10,...,0.036122,2,0.150142,0.087967,0.286819,0.080736,0.249670,0.000000,0.040309,0.104357
998,668d5331aa58b7e75ff160d7,0.091113,0.161720,0.020506,0.714286,63,234906_experiment_2024-07-10_19h00.46.327.csv,0.609375,"The taller gnomes perform better, yellow is th...",10,...,0.028196,1,0.048805,0.216708,0.191430,0.261326,0.135676,0.117352,0.028703,0.000000
999,668d822d705c807ed4664ae1,0.820187,0.642993,0.997381,2.688492,5,564113_experiment_2024-07-10_18h59.42.430.csv,0.730469,There are two types of baskets RED or YELLOWRe...,10,...,0.095384,2,0.133503,0.027035,0.362690,0.073281,0.287345,0.015535,0.000000,0.100610


Correlation table of Topics

In [4]:
var_names = ["score_corrected", "pl2_understanding", "pl1_understanding"] + [f'topic_{n}' for n in range(len(topic_distr[0, :]))] + [f'parent_topic_{n}' for n in range(len(topic_distr[0, :]))]

cor_table = df_stat[var_names].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(cor_table, annot=False, cmap='coolwarm', vmin=-1, vmax=1, center=0)
plt.title('Correlation Heatmap')
plt.show()

NameError: name 'topic_distr' is not defined

Save data with topic info

In [ ]:
#df_stat.to_csv("../results/NLP/data_topic_weights.csv", header=True, index=False)

Save data for plot

In [ ]:
from umap import UMAP
from typing import List, Union

topic_per_doc = topic_model.topics_
sample = 1

indices = []
for topic in set(topic_per_doc):
    s = np.where(np.array(topic_per_doc) == topic)[0]
    size = len(s) if len(s) < 100 else int(len(s) * sample)
    indices.extend(np.random.choice(s, size=size, replace=False))
indices = np.array(indices)

df = pd.DataFrame({"topic": np.array(topic_per_doc)[indices]})
df["doc"] = [docs[index] for index in indices]
df["topic"] = [topic_per_doc[index] for index in indices]

# Extract embeddings if not already done
embeddings_to_reduce = topic_model._extract_embeddings(df.doc.to_list(), method="document")

# Reduce input embeddings
umap_model = UMAP(n_neighbors=10, n_components=2, min_dist=0.0, metric="cosine").fit(embeddings_to_reduce)
embeddings_2d = umap_model.embedding_

unique_topics = set(topic_per_doc)
topics = unique_topics

# Combine data
df["x"] = embeddings_2d[:, 0]
df["y"] = embeddings_2d[:, 1]

df.to_csv("../results/NLP/embedding_plot_data.csv", index=False)